# Likelihood geometry for pairs of node heights

The tree normalising flow exists to capture dependence between node heights that
a mean-field approximation cannot. This notebook looks at that dependence
directly, in the likelihood rather than through a fitted posterior: fix every
parameter at (close to) its maximum, then vary **pairs** of node heights on a
grid and plot the surface.

Three things it answers:

1. **Which pairs are actually coupled, and how does that change with sequence
   length?** Pairs involving the root are included, since the root height is the
   coordinate the existing `root_full_rank` approximation singles out.
2. **How much of the coupling survives the node-height ratio transform?** Every
   surface is drawn twice -- once over the node heights themselves, and once over
   the *unconstrained coordinates* the approximations (and the flow) actually
   work in. The ratio transform exists to absorb the ordering constraint, and
   these plots show how much correlation it absorbs with it.
3. **Where does a relaxed clock put the dependence?** With per-branch rates, a
   branch length is a product of a rate and a time, so heights and rates trade
   off against each other. The last section plots those surfaces and the
   posterior correlations between node heights and branch rates.

This complements `tree_normalizing_flow.ipynb`: that notebook measures how well
each variational family fits; this one shows what there is to fit.

## Setup

In [ ]:
import os
from functools import partial

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from tqdm import tqdm

tf.get_logger().setLevel("ERROR")
tfd = tfp.distributions

from treeflow import DEFAULT_FLOAT_DTYPE_TF
from treeflow.bijectors.node_height_ratio_bijector import NodeHeightRatioChainBijector
from treeflow.distributions import (
    DiscreteParameterMixture,
    DiscretizedDistribution,
    LeafCTMC,
)
from treeflow.distributions.tree.coalescent.constant_coalescent import (
    ConstantCoalescent,
)
from treeflow.evolution.substitution import HKY, get_transition_probabilities_tree
from treeflow.mcmc import (
    check_effective_sample_size,
    effective_sample_size_summary,
    fit_fixed_topology_random_walk_metropolis,
)
from treeflow.model.ml import fit_fixed_topology_maximum_likelihood_sgd
from treeflow.model.phylo_model import PhyloModel
from treeflow.traversal.anchor_heights import get_anchor_heights_tensor
from treeflow.tree.rooted.tensorflow_rooted_tree import convert_tree_to_tensor

# The benchmark pipeline's simulator (importable because the notebook's own
# directory is the kernel's working directory).
from benchmarks.simulate import (
    simulate_alignment,
    simulate_coalescent_tree,
    simulate_sampling_times,
)

In [ ]:
_env = lambda name, default: type(default)(os.environ.get(name, default))

TAXON_COUNT = _env("TREEFLOW_GEOMETRY_TAXON_COUNT", 12)
SAMPLING_WINDOW = _env("TREEFLOW_GEOMETRY_SAMPLING_WINDOW", 5.0)
DATA_SEED = _env("TREEFLOW_GEOMETRY_DATA_SEED", 7)
SITE_COUNTS = [
    int(n)
    for n in os.environ.get("TREEFLOW_GEOMETRY_SITE_COUNTS", "200,1000,5000").split(",")
]
GRID = _env("TREEFLOW_GEOMETRY_GRID", 30)          # points per axis
MAP_STEPS = _env("TREEFLOW_GEOMETRY_MAP_STEPS", 3_000)
MCMC_RESULTS = _env("TREEFLOW_GEOMETRY_MCMC_RESULTS", 5_000)
MCMC_BURNIN = _env("TREEFLOW_GEOMETRY_MCMC_BURNIN", 5_000)
MCMC_CHAINS = _env("TREEFLOW_GEOMETRY_MCMC_CHAINS", 4)

DTYPE = DEFAULT_FLOAT_DTYPE_TF
constant = lambda x: tf.constant(x, dtype=DTYPE)
SITE_CATEGORY_COUNT = 4
progress = partial(tqdm, mininterval=15.0, ncols=72)
print(f"{TAXON_COUNT} taxa | sequence lengths {SITE_COUNTS} | {GRID}x{GRID} grids")

In [ ]:
TRUE_PARAMETERS = dict(
    pop_size=5.0,
    clock_rate=0.05,
    kappa=3.0,
    frequencies=[0.3, 0.2, 0.2, 0.3],
    site_gamma_shape=0.8,
)

data_rng = np.random.default_rng(DATA_SEED)
sampling_times = simulate_sampling_times(TAXON_COUNT, SAMPLING_WINDOW, data_rng)
tree = convert_tree_to_tensor(
    simulate_coalescent_tree(sampling_times, TRUE_PARAMETERS["pop_size"], data_rng)
)
subst_model = HKY()
node_count = TAXON_COUNT - 1
ROOT = node_count - 1  # the root is the last internal node

simulation_model = PhyloModel(
    dict(
        tree=dict(coalescent=dict(pop_size=TRUE_PARAMETERS["pop_size"])),
        clock=dict(strict=dict(clock_rate=TRUE_PARAMETERS["clock_rate"])),
        site=dict(
            discrete_gamma=dict(
                site_gamma_shape=TRUE_PARAMETERS["site_gamma_shape"],
                category_count=SITE_CATEGORY_COUNT,
            )
        ),
        substitution=dict(
            hky=dict(
                kappa=TRUE_PARAMETERS["kappa"],
                frequencies=TRUE_PARAMETERS["frequencies"],
            )
        ),
    )
)
# One alignment per sequence length, all down the same tree.
alignments = {
    sites: constant(simulate_alignment(tree, simulation_model, sites, DATA_SEED))
    for sites in SITE_COUNTS
}
true_heights = tree.node_heights.numpy()
print("true node heights:", np.round(true_heights, 2))

### The models

A strict clock (one rate for the whole tree) and a relaxed clock (an independent
lognormal rate per branch). The data are simulated under the strict clock; the
relaxed-clock *analysis* of the same data is what exposes the height-rate
trade-off, which is a property of the model rather than of how the data arose.

In [ ]:
def sequence_distribution(tree_, rates, kappa, frequencies, site_gamma_shape, sites):
    """The phylogenetic likelihood term. ``rates`` is a scalar clock rate or one
    rate per branch."""
    distance_tree = tree_.get_unrooted_tree() * rates
    site_rate_distribution = DiscretizedDistribution(
        category_count=SITE_CATEGORY_COUNT,
        distribution=tfd.Gamma(site_gamma_shape, site_gamma_shape),
    )
    transition_probs = get_transition_probabilities_tree(
        distance_tree,
        subst_model,
        rate_categories=site_rate_distribution.normalised_support,
        frequencies=frequencies,
        kappa=kappa,
    )
    leaf_ctmc = LeafCTMC(
        transition_probs, tf.expand_dims(frequencies, -2), use_native=True
    )
    return tfd.Sample(
        DiscreteParameterMixture(site_rate_distribution, leaf_ctmc),
        sample_shape=sites,
    )


def strict_clock_model(sites):
    def alignment_dist(tree, clock_rate, kappa, frequencies, site_gamma_shape):
        return sequence_distribution(
            tree,
            clock_rate[..., tf.newaxis],
            kappa,
            frequencies,
            site_gamma_shape,
            sites,
        )

    return tfd.JointDistributionNamed(
        dict(
            pop_size=tfd.LogNormal(constant(1.0), constant(1.5)),
            clock_rate=tfd.LogNormal(constant(-2.0), constant(2.0)),
            kappa=tfd.LogNormal(constant(1.0), constant(1.25)),
            frequencies=tfd.Dirichlet(constant([2.0, 2.0, 2.0, 2.0])),
            site_gamma_shape=tfd.LogNormal(constant(0.0), constant(1.0)),
            tree=lambda pop_size: ConstantCoalescent(
                tree.taxon_count, pop_size, tree.sampling_times, tree_name="tree"
            ),
            alignment=alignment_dist,
        )
    ).experimental_pin(alignment=alignments[sites])


def relaxed_clock_model(sites):
    branch_count = 2 * TAXON_COUNT - 2

    def alignment_dist(tree, branch_rates, kappa, frequencies, site_gamma_shape):
        return sequence_distribution(
            tree, branch_rates, kappa, frequencies, site_gamma_shape, sites
        )

    return tfd.JointDistributionNamed(
        dict(
            pop_size=tfd.LogNormal(constant(1.0), constant(1.5)),
            branch_rates=tfd.Sample(
                tfd.LogNormal(constant(np.log(TRUE_PARAMETERS["clock_rate"])), constant(0.4)),
                branch_count,
            ),
            kappa=tfd.LogNormal(constant(1.0), constant(1.25)),
            frequencies=tfd.Dirichlet(constant([2.0, 2.0, 2.0, 2.0])),
            site_gamma_shape=tfd.LogNormal(constant(0.0), constant(1.0)),
            tree=lambda pop_size: ConstantCoalescent(
                tree.taxon_count, pop_size, tree.sampling_times, tree_name="tree"
            ),
            alignment=alignment_dist,
        )
    ).experimental_pin(alignment=alignments[sites])

### The point everything is fixed at

`fit_fixed_topology_maximum_likelihood_sgd` maximises the model's *unnormalised
posterior*, so strictly this is a MAP rather than an MLE; with these priors and
this much data the difference is small, and it is the right point to expand
around for a notebook about posterior geometry.

In [ ]:
def fit_map(model, init, num_steps=MAP_STEPS):
    variables, trace, _ = fit_fixed_topology_maximum_likelihood_sgd(
        model,
        topologies=dict(tree=tree.topology),
        num_steps=num_steps,
        optimizer=tf.optimizers.Adam(0.02),
        init=init,
        progress_bar=progress,
    )
    return variables, float(np.asarray(trace.log_likelihood).reshape(-1)[-1])


strict_map = {}
for sites in SITE_COUNTS:
    variables, value = fit_map(strict_clock_model(sites), init=dict(tree=tree))
    strict_map[sites] = variables
    print(f"{sites:5d} sites: log posterior {value:10.2f} | "
          f"clock {float(variables['clock_rate']):.4f} | "
          f"root height {float(variables['tree'].node_heights[-1]):.3f}")

## Surfaces over pairs of node heights

Everything is held at the MAP except the two coordinates being varied. Two
coordinate systems:

* **node heights** -- the natural parameterisation. Combinations violating the
  ordering constraint (a node below its children, or above its parent) are not
  trees at all; those cells are masked, and the shape of the masked region is
  itself part of the geometry.
* **unconstrained coordinates** -- what
  `NodeHeightRatioChainBijector` maps to heights, and what every approximation in
  treeflow actually parameterises. Every point in this space is a valid tree, so
  there is no masked region; a diagonal ridge here is dependence a mean-field
  approximation would miss.

In [ ]:
anchor_heights = get_anchor_heights_tensor(tree.topology, tree.sampling_times)
height_bijector = NodeHeightRatioChainBijector(tree.topology, anchor_heights)


def evaluate_in_chunks(evaluate, mesh, sites, budget_bytes=4e8):
    """Evaluate a grid of trees in memory-bounded chunks.

    The likelihood materialises partials of shape
    [batch, site, rate category, node, state], so a whole grid at once is tens of
    gigabytes at a long alignment -- enough to take the machine down rather than
    raise. The chunk size is set from that product against a fixed budget.
    """
    per_point = sites * SITE_CATEGORY_COUNT * (2 * TAXON_COUNT - 1) * 4 * 8
    chunk = max(1, int(budget_bytes // max(per_point, 1)))
    return np.concatenate(
        [
            np.asarray(evaluate(constant(mesh[start : start + chunk])))
            for start in range(0, len(mesh), chunk)
        ]
    )


def log_posterior_fn(model, variables, sites):
    """Vectorised log posterior as a function of a batch of node heights."""
    del sites

    def evaluate(node_heights):
        values = dict(variables)
        values["tree"] = variables["tree"].with_node_heights(node_heights)
        return model.unnormalized_log_prob(values)

    return evaluate


def feasible_interval(heights, node, margin=0.02, root_span=0.6):
    """The interval a node's height can occupy with the others held fixed.

    Bounded below by its tallest child (a sampling time, for a leaf child) and
    above by its parent; the root is unbounded above, so it gets a window
    proportional to its own height. Using this rather than a fixed percentage
    keeps a grid inside the region where the tree is a tree, which matters for
    nodes wedged between a close child and a close parent.
    """
    node_child_ids = tree.topology.node_child_indices.numpy()[node]
    child_heights = [
        heights[int(child) - TAXON_COUNT]
        if child >= TAXON_COUNT
        else float(tree.sampling_times.numpy()[int(child)])
        for child in node_child_ids
    ]
    lower = max(child_heights)
    upper = (
        heights[node] * (1 + root_span)
        if node == ROOT
        else heights[int(parent_of[node])]
    )
    width = upper - lower
    return lower + margin * width, upper - margin * width


def height_grid(centre, i, j):
    """A [GRID*GRID, node] batch varying coordinates i and j about ``centre``."""
    axis_i = np.linspace(*feasible_interval(centre, i), GRID)
    axis_j = np.linspace(*feasible_interval(centre, j), GRID)
    mesh = np.tile(centre, (GRID * GRID, 1))
    mesh[:, i] = np.repeat(axis_i, GRID)
    mesh[:, j] = np.tile(axis_j, GRID)
    return axis_i, axis_j, mesh


def unconstrained_grid(centre, i, j, span=1.5):
    axis_i = np.linspace(centre[i] - span, centre[i] + span, GRID)
    axis_j = np.linspace(centre[j] - span, centre[j] + span, GRID)
    mesh = np.tile(centre, (GRID * GRID, 1))
    mesh[:, i] = np.repeat(axis_i, GRID)
    mesh[:, j] = np.tile(axis_j, GRID)
    return axis_i, axis_j, mesh


def surface(model, variables, sites, i, j, unconstrained):
    """Log posterior over a pair of coordinates, relative to its maximum."""
    evaluate = log_posterior_fn(model, variables, sites)
    heights = variables["tree"].node_heights.numpy()
    if unconstrained:
        centre = height_bijector.inverse(constant(heights)).numpy()
        axis_i, axis_j, mesh = unconstrained_grid(centre, i, j)
        mesh = height_bijector.forward(constant(mesh)).numpy()
        feasible = np.ones((GRID, GRID), dtype=bool)
    else:
        axis_i, axis_j, mesh = height_grid(heights, i, j)
        branch_lengths = tree.with_node_heights(constant(mesh)).branch_lengths.numpy()
        feasible = (branch_lengths > 0).all(-1).reshape(GRID, GRID)
    values = evaluate_in_chunks(evaluate, mesh, sites).reshape(GRID, GRID)
    values = np.where(feasible & np.isfinite(values), values, np.nan)
    return axis_i, axis_j, values - np.nanmax(values)

In [ ]:
def implied_correlation(values, axis_i, axis_j):
    """Correlation of the Gaussian that matches the surface's curvature.

    Fits a quadratic to the (finite) cells and reads the correlation off the
    fitted Hessian -- the number a Gaussian approximation would have to match.
    """
    grid_i, grid_j = np.meshgrid(axis_i, axis_j, indexing="ij")
    mask = np.isfinite(values)
    if mask.sum() < 12:
        return np.nan
    x = (grid_i[mask] - np.mean(axis_i)) / np.std(axis_i)
    y = (grid_j[mask] - np.mean(axis_j)) / np.std(axis_j)
    design = np.stack([np.ones_like(x), x, y, x**2, y**2, x * y], axis=-1)
    coefficients, *_ = np.linalg.lstsq(design, values[mask], rcond=None)
    _, _, _, a, b, c = coefficients
    hessian = np.array([[2 * a, c], [c, 2 * b]])
    if np.linalg.det(hessian) <= 0 or a >= 0 or b >= 0:
        return np.nan  # not a well-defined peak in this window
    covariance = np.linalg.inv(-hessian)
    return covariance[0, 1] / np.sqrt(covariance[0, 0] * covariance[1, 1])

In [ ]:
# Which pairs to look at. Node ids are internal-node indices; the root is last.
parent_of = tree.topology.parent_indices.numpy()[TAXON_COUNT:] - TAXON_COUNT
children_of = tree.topology.node_child_indices.numpy() - TAXON_COUNT

root_children = [int(c) for c in children_of[ROOT] if c >= 0]
deepest_non_root = int(np.argmin(true_heights[:-1]))
adjacent_pair = next(
    (int(child), int(parent)) for child, parent in enumerate(parent_of)
    if parent != ROOT and child != ROOT
)
def ancestors(node):
    """Internal-node ancestors of ``node``, root included."""
    path = []
    while node != ROOT:
        node = int(parent_of[node])
        path.append(node)
    return set(path)


# A genuinely non-adjacent pair: neither node an ancestor of the other, so the
# only route between them is through their common ancestor.
cousin_pair = next(
    (i, j)
    for i in range(node_count)
    for j in range(i + 1, node_count)
    if j not in ancestors(i) and i not in ancestors(j)
)

PAIRS = {
    f"parent-child ({adjacent_pair[0]}, {adjacent_pair[1]})": adjacent_pair,
    f"root and its child ({root_children[0]}, {ROOT})": (root_children[0], ROOT),
    f"root and a deep node ({deepest_non_root}, {ROOT})": (deepest_non_root, ROOT),
    f"non-adjacent pair {cousin_pair}": cousin_pair,
}
for label, (i, j) in PAIRS.items():
    print(f"{label:34s} heights {true_heights[i]:6.2f} {true_heights[j]:6.2f}")

In [ ]:
def plot_surfaces(models, maps, unconstrained, title):
    fig, axs = plt.subplots(
        len(PAIRS), len(SITE_COUNTS),
        figsize=(4 * len(SITE_COUNTS), 3.2 * len(PAIRS)),
        squeeze=False,
    )
    correlations = {}
    for row, (label, (i, j)) in enumerate(PAIRS.items()):
        for column, sites in enumerate(SITE_COUNTS):
            axis_i, axis_j, values = surface(
                models[sites], maps[sites], sites, i, j, unconstrained
            )
            correlations[(label, sites)] = implied_correlation(values, axis_i, axis_j)
            ax = axs[row][column]
            levels = np.linspace(-20, 0, 21)
            contours = ax.contourf(
                axis_j, axis_i, np.clip(values, -20, 0), levels=levels, cmap="viridis"
            )
            ax.set_facecolor("0.85")  # infeasible cells show through
            if np.isfinite(values).any():
                peak = np.unravel_index(np.nanargmax(values), values.shape)
                ax.plot(axis_j[peak[1]], axis_i[peak[0]], "r+", ms=10)
            if row == 0:
                ax.set_title(f"{sites} sites")
            if column == 0:
                ax.set_ylabel(label, fontsize=8)
            ax.tick_params(labelsize=7)
    fig.colorbar(contours, ax=axs, shrink=0.6, label="log posterior - max")
    fig.suptitle(title, y=0.995)
    return pd.Series(correlations).unstack()


height_correlations = plot_surfaces(
    {sites: strict_clock_model(sites) for sites in SITE_COUNTS},
    strict_map,
    unconstrained=False,
    title="Strict clock: pairs of node heights (grey = not a valid tree)",
)

In [ ]:
unconstrained_correlations = plot_surfaces(
    {sites: strict_clock_model(sites) for sites in SITE_COUNTS},
    strict_map,
    unconstrained=True,
    title="Strict clock: the same pairs in unconstrained coordinates",
)

In [ ]:
comparison = pd.concat(
    {"node heights": height_correlations, "unconstrained": unconstrained_correlations},
    axis=1,
)
comparison.round(3)

### Strict clock findings

*(Filled in from the run above.)*

## Relaxed clock: node heights against branch rates

With one rate per branch, a branch length is `rate x time`: the data constrain
the product, so a node height and the rate on an adjacent branch trade off
directly. This is dependence between the tree block and a *per-branch* parameter
-- exactly what the flow's per-node conditioner is for.

In [ ]:
relaxed_models = {sites: relaxed_clock_model(sites) for sites in SITE_COUNTS}
relaxed_map = {}
for sites in SITE_COUNTS:
    variables, value = fit_map(
        relaxed_models[sites],
        init=dict(
            tree=tree,
            branch_rates=constant(
                np.full(2 * TAXON_COUNT - 2, TRUE_PARAMETERS["clock_rate"])
            ),
        ),
    )
    relaxed_map[sites] = variables
    print(f"{sites:5d} sites: log posterior {value:10.2f} | "
          f"branch rates {float(tf.reduce_min(variables['branch_rates'])):.4f}"
          f"-{float(tf.reduce_max(variables['branch_rates'])):.4f}")

In [ ]:
def rate_height_surface(model, variables, node, branch, span=0.35):
    """Log posterior over (height of ``node``, rate of ``branch``)."""
    heights = variables["tree"].node_heights.numpy()
    rates = np.asarray(variables["branch_rates"])
    axis_height = np.linspace(*feasible_interval(heights, node), GRID)
    axis_rate = np.linspace(rates[branch] * (1 - span), rates[branch] * (1 + span), GRID)
    height_mesh = np.tile(heights, (GRID * GRID, 1))
    rate_mesh = np.tile(rates, (GRID * GRID, 1))
    height_mesh[:, node] = np.repeat(axis_height, GRID)
    rate_mesh[:, branch] = np.tile(axis_rate, GRID)

    values = dict(variables)
    values["tree"] = variables["tree"].with_node_heights(constant(height_mesh))
    values["branch_rates"] = constant(rate_mesh)
    surface_values = np.asarray(model.unnormalized_log_prob(values)).reshape(GRID, GRID)
    feasible = (
        (tree.with_node_heights(constant(height_mesh)).branch_lengths.numpy() > 0)
        .all(-1)
        .reshape(GRID, GRID)
    )
    surface_values = np.where(
        feasible & np.isfinite(surface_values), surface_values, np.nan
    )
    return axis_height, axis_rate, surface_values - np.nanmax(surface_values)


# The branch above node ``i`` is branch ``taxon_count + i``; pick a distant one
# for contrast.
focus_node = ROOT - 1
adjacent_branch = TAXON_COUNT + focus_node
distant_branch = int(np.argmax(np.abs(np.arange(2 * TAXON_COUNT - 2) - adjacent_branch)))
RATE_PAIRS = {
    f"node {focus_node} and its own branch": (focus_node, adjacent_branch),
    f"node {focus_node} and a distant branch": (focus_node, distant_branch),
    f"root and a root branch": (ROOT, TAXON_COUNT + int(np.asarray(children_of[ROOT])[0])),
}

fig, axs = plt.subplots(
    len(RATE_PAIRS), len(SITE_COUNTS),
    figsize=(4 * len(SITE_COUNTS), 3.2 * len(RATE_PAIRS)),
    squeeze=False,
)
rate_correlations = {}
for row, (label, (node, branch)) in enumerate(RATE_PAIRS.items()):
    for column, sites in enumerate(SITE_COUNTS):
        axis_height, axis_rate, values = rate_height_surface(
            relaxed_models[sites], relaxed_map[sites], node, branch, sites
        )
        rate_correlations[(label, sites)] = implied_correlation(
            values, axis_height, axis_rate
        )
        ax = axs[row][column]
        contours = ax.contourf(
            axis_rate, axis_height, np.clip(values, -20, 0),
            levels=np.linspace(-20, 0, 21), cmap="magma",
        )
        ax.set_facecolor("0.85")
        peak = np.unravel_index(np.nanargmax(values), values.shape)
        ax.plot(axis_rate[peak[1]], axis_height[peak[0]], "c+", ms=10)
        if row == 0:
            ax.set_title(f"{sites} sites")
        if column == 0:
            ax.set_ylabel(label, fontsize=8)
        ax.set_xlabel("branch rate", fontsize=7)
        ax.tick_params(labelsize=7)
fig.colorbar(contours, ax=axs, shrink=0.6, label="log posterior - max")
fig.suptitle("Relaxed clock: node height against branch rate", y=0.995)
pd.Series(rate_correlations).unstack().round(3)

### Posterior correlations between node heights and branch rates

The surfaces above hold every other coordinate fixed, which shows the *local*
trade-off. The posterior correlation additionally accounts for everything else
moving, so it is the number a variational family would have to reproduce. This
samples the relaxed-clock posterior with the random-walk reference sampler and
reports its diagnostics -- a relaxed clock adds one dimension per branch, so the
effective sample size matters even more here than usual.

In [ ]:
reference = fit_fixed_topology_random_walk_metropolis(
    relaxed_models[SITE_COUNTS[-1]],
    topologies=dict(tree=tree.topology),
    num_results=MCMC_RESULTS,
    num_burnin_steps=MCMC_BURNIN,
    num_chains=MCMC_CHAINS,
    thin=5,
    init_state=dict(tree=tree, branch_rates=relaxed_map[SITE_COUNTS[-1]]["branch_rates"]),
    progress_bar=progress,
    seed=(31, 31),
)
passed, ess_summary = check_effective_sample_size(reference, min_ess=200.0)
print(f"acceptance {float(reference.acceptance_rate):.3f}, "
      f"ESS check {'PASSED' if passed else 'FAILED'}")
pd.DataFrame(ess_summary).T.round(1)

In [ ]:
flatten = lambda x: np.asarray(x).reshape((-1,) + np.asarray(x).shape[2:])
posterior_heights = flatten(reference.samples["tree"].node_heights)
posterior_rates = flatten(reference.samples["branch_rates"])

correlation = np.corrcoef(
    np.concatenate([posterior_heights, np.log(posterior_rates)], axis=-1).T
)
cross = correlation[:node_count, node_count:]

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
image = axs[0].imshow(cross, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
axs[0].set_xlabel("branch (log rate)")
axs[0].set_ylabel("internal node (height)")
axs[0].set_title("Posterior correlation, node heights x branch rates")
plt.colorbar(image, ax=axs[0], fraction=0.046)

# Is the correlation concentrated on a node's own branches?
branch_count = 2 * TAXON_COUNT - 2
own_branch = np.zeros_like(cross, dtype=bool)
node_child_ids = tree.topology.node_child_indices.numpy()
for node in range(node_count):
    above = TAXON_COUNT + node
    if above < branch_count:  # the root has no branch above it
        own_branch[node, above] = True
    for child in node_child_ids[node]:  # the branches below it
        own_branch[node, int(child)] = True

axs[1].hist(np.abs(cross[own_branch]), bins=20, alpha=0.7, density=True,
            label="a node's own branches")
axs[1].hist(np.abs(cross[~own_branch]), bins=20, alpha=0.7, density=True,
            label="all other pairs")
axs[1].set_xlabel("|correlation|")
axs[1].set_ylabel("density")
axs[1].set_title("Height-rate correlation, adjacent vs distant")
axs[1].legend(fontsize=8)
plt.tight_layout()

pd.Series({
    "mean |corr|, a node's own branches": np.abs(cross[own_branch]).mean(),
    "mean |corr|, all other pairs": np.abs(cross[~own_branch]).mean(),
    "max |corr| overall": np.abs(cross).max(),
}).round(3).to_frame("height x rate correlation")

### Relaxed clock findings

*(Filled in from the run above.)*

## What this means for the approximations

*(Filled in from the run above.)*